# Peer Review Topology: arXiv Paper Discovery and Verification

This notebook demonstrates the **peer_review** multi-agent topology using a simple research task: finding relevant arXiv papers on the **H0 tension** (Hubble constant tension) and verifying that the references are valid.

**How it works:**
- **Architect Agent**: Searches for and proposes a list of relevant arXiv papers
- **Reviewer Agent**: Verifies that the proposed papers exist and the arXiv IDs are correctly formatted
- **Iteration**: The agents iterate up to 4 rounds until the reviewer approves (says "APPROVED") or the maximum rounds are reached

This pattern is useful for research literature review, fact-checking, and reference validation tasks.

## 1. Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path
from IPython.display import Markdown, display

# If the notebook lives in examples/, make sure src/ is importable
sys.path.insert(0, str(Path.cwd().parent / "src"))

from mas_topologies import TopologyAgent, PromptSet

print("Imports successful!")

## 2. API Configuration

Set your API key before running. We'll use **Claude** models as they're well-suited for research and literature tasks.

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
# or
export OPENAI_API_KEY="sk-..."
```

In [ ]:
# Set your API keys here (or export them in your environment)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
def build_client(provider="auto"):
    """Build a model client. Prefers Anthropic (Claude) for research tasks."""
    if provider in ("anthropic", "auto") and os.environ.get("ANTHROPIC_API_KEY"):
        from autogen_ext.models.anthropic import AnthropicChatCompletionClient
        model = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-sonnet-20241022")
        client = AnthropicChatCompletionClient(
            model=model, 
            api_key=os.environ["ANTHROPIC_API_KEY"]
        )
        return client, f"anthropic/{model}"

    if provider in ("openai", "auto") and os.environ.get("OPENAI_API_KEY"):
        from autogen_ext.models.openai import OpenAIChatCompletionClient
        model = os.environ.get("OPENAI_MODEL", "gpt-4o")
        client = OpenAIChatCompletionClient(
            model=model, 
            api_key=os.environ["OPENAI_API_KEY"]
        )
        return client, f"openai/{model}"

    raise EnvironmentError(
        "No API key found. Export ANTHROPIC_API_KEY or OPENAI_API_KEY."
    )

MODEL_CLIENT, PROVIDER_LABEL = build_client("auto")
print(f"Using provider: {PROVIDER_LABEL}")

## 3. Define Agent Prompts

We'll define two roles:
- **Architect**: Finds relevant arXiv papers on a given topic
- **Reviewer**: Verifies that the papers exist and the references are valid

In [ ]:
ARCHITECT_PROMPT = """You are an expert astrophysics research librarian with deep knowledge of cosmology literature.

Your task is to find the most relevant and recent arXiv papers on a given topic.

**Requirements:**
1. Provide 5-8 highly relevant papers
2. For each paper, include:
   - arXiv ID (format: YYMM.NNNNN or YYMM.NNNNNVV)
   - Title
   - Authors (first author + "et al." if multiple)
   - Brief one-sentence summary of relevance
3. Prioritize papers from the last 3-5 years
4. Focus on high-impact work (well-cited, influential authors, major collaborations)
5. Ensure arXiv IDs are real and correctly formatted

**Output Format:**
For each paper, use this exact format:

```
arXiv:YYMM.NNNNN
Title: [Full paper title]
Authors: [First author et al.]
Summary: [One sentence explaining why this paper is relevant]
```

Be precise with arXiv IDs - they must be real papers that exist on arxiv.org."""

REVIEWER_PROMPT = """You are a meticulous research verification specialist.

Your task is to verify that the arXiv references provided are valid and correctly formatted.

**Verification Checklist:**
1. ✓ Each arXiv ID follows the correct format (YYMM.NNNNN or YYMM.NNNNNVV)
2. ✓ arXiv IDs look plausible (reasonable year/month, reasonable paper number)
3. ✓ Each entry has Title, Authors, and Summary
4. ✓ The papers appear to be relevant to the requested topic
5. ✓ No obvious duplicates
6. ✓ Authors and titles look realistic (not obviously fabricated)

**Your Response:**
- If ALL references pass verification, respond with exactly: **APPROVED**
- If you find ANY issues, list them clearly and ask the Architect to provide corrected references

**Common Issues to Check:**
- Invalid arXiv ID format (e.g., wrong number of digits)
- Implausible dates (e.g., arXiv IDs from before arXiv existed or far in the future)
- Missing information (title, authors, or summary)
- Off-topic papers
- Obvious formatting errors

Be thorough but reasonable. Focus on catching clear errors rather than nitpicking."""

# Create the PromptSet
PROMPT_SET = PromptSet(
    system_message=ARCHITECT_PROMPT,
    critic_message=REVIEWER_PROMPT,
)

print("Prompts configured for peer_review topology")

## 4. Create the Peer Review Agent

We'll create a `TopologyAgent` configured with the `peer_review` topology. This will orchestrate the Architect and Reviewer in an iterative loop.

In [ ]:
# Create logs directory
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

# Create the topology agent
agent = TopologyAgent(
    model_client=MODEL_CLIENT,
    topology="peer_review",
    prompts=PROMPT_SET,
    log_dir=str(LOG_DIR),
)

print(f"Peer review agent created")
print(f"Session logs will be saved to: {LOG_DIR}")

## 5. Define the Research Task

We'll ask the agents to find papers on the **H0 tension** - the discrepancy between different measurements of the Hubble constant, which is one of the most important open problems in modern cosmology.

In [ ]:
RESEARCH_CONTEXT = """Find the most relevant and recent arXiv papers on the Hubble constant tension (H0 tension).

The H0 tension refers to the significant discrepancy between measurements of the Hubble constant 
from the early universe (CMB observations from Planck) versus late-universe measurements 
(local distance ladder from supernovae and Cepheid variables).

Focus on:
- Papers addressing the tension itself
- Proposed theoretical solutions or explanations
- New observational constraints or measurements
- Recent reviews or perspective papers

Provide 5-8 key papers from the last 3-5 years."""

print("Research task defined:")
print("-" * 80)
print(RESEARCH_CONTEXT)
print("-" * 80)

## 6. Run the Peer Review Topology

Now we'll run the agent. You'll see:
1. The Architect proposes a list of papers
2. The Reviewer checks them
3. If issues are found, the Architect revises
4. This continues until APPROVED or max 4 rounds

In [ ]:
result = await agent.run(
    RESEARCH_CONTEXT,
    metadata={
        "task": "arxiv_h0_tension",
        "topology": "peer_review",
    }
)

print(f"\n{'='*80}")
print(f"Peer review complete!")
print(f"Total rounds used: {result.rounds_used}")
print(f"{'='*80}")

## 7. View Results

Let's examine the final approved paper list and the conversation history:

In [ ]:
# Display the final approved response
print("FINAL APPROVED PAPER LIST:")
print("="*80)
display(Markdown(result.response_text))

In [ ]:
# Show the conversation history
print("\n" + "="*80)
print("CONVERSATION HISTORY:")
print("="*80)

for i, msg in enumerate(result.messages, 1):
    source = msg["source"]
    content = msg["content"]
    
    print(f"\n--- Message {i}: {source} ---")
    print(content[:500] + "..." if len(content) > 500 else content)
    print()

## 8. Inspect Session Logs

The session log captures the full interaction in JSON format for reproducibility and analysis:

In [ ]:
import json

# Find the most recent log file
log_files = sorted(LOG_DIR.glob("session_*.jsonl"))
if log_files:
    latest_log = log_files[-1]
    print(f"Reading session log: {latest_log}")
    
    with open(latest_log, "r") as f:
        session_data = json.loads(f.read())
    
    print("\nSession metadata:")
    print(f"  Topology: {session_data.get('topology')}")
    print(f"  Rounds used: {session_data.get('rounds_used')}")
    print(f"  Timestamp: {session_data.get('timestamp')}")
    print(f"  Task metadata: {session_data.get('metadata')}")
else:
    print("No log files found")

## Summary

This notebook demonstrated the **peer_review** topology for research tasks:

**Key takeaways:**
- The Architect agent proposes arXiv papers relevant to the H0 tension
- The Reviewer agent verifies references and checks for errors
- The agents iterate until the Reviewer approves or max rounds are reached
- All interactions are logged for reproducibility

**Potential extensions:**
- Try different research topics
- Adjust the number of papers requested
- Modify verification criteria
- Compare with other topologies (single, adversarial, parallel_ensemble)
- Add actual arXiv API calls to verify papers exist programmatically